# TMDb Data Profile

This notebook documents the data-quality checks performed on the candidate movie list and the TMDb responses. It records the issues found, their effect on movie coverage, and the evidence behind decisions used in the pipeline.

## 1. Candidate ID quality

The supplied CSV contains one column of candidate TMDb movie IDs. The first check accounts for every input row and identifies which distinct IDs can be sent to TMDb.

In [88]:
from pathlib import Path

import pandas as pd

# Supports starting the notebook from the project root or Data Profile.
project_root = Path.cwd()
if project_root.name.lower() == "data profile":
    project_root = project_root.parent

audit_path = project_root / "data" / "validated" / "id_audit.csv"
id_audit = pd.read_csv(audit_path, dtype={"tmdb_id": str}, keep_default_na=False)

status_order = ["missing", "invalid_format", "duplicate", "ready_to_fetch"]
counts = id_audit["status"].value_counts().reindex(status_order, fill_value=0)

assert counts.sum() == len(id_audit), "The audit has an unexpected status."

print(f"Total input rows: {len(id_audit)}")
display(counts.rename("rows").to_frame())

Total input rows: 127


,rows
status,
missing,1
invalid_format,3
duplicate,6
ready_to_fetch,117


In [89]:
issues = id_audit.loc[
    id_audit["status"].ne("ready_to_fetch"),
    ["source_line", "tmdb_id", "status"],
].copy()

# Quotes make the blank value visible as '' in the notebook.
issues["tmdb_id"] = issues["tmdb_id"].map(repr)
display(issues)

,source_line,tmdb_id,status
28,30,'-1',invalid_format
41,43,'603',duplicate
58,60,'278',duplicate
67,69,'',missing
88,90,'1891',duplicate
89,91,'155',duplicate
92,94,'550',duplicate
93,95,'tt0468569',invalid_format
100,102,'680',duplicate
109,111,'N/A',invalid_format


### Finding and decision

Of the 127 input rows, 1 was blank, 3 failed the positive-integer rule,
and 6 were repeat occurrences of IDs already present in the list.
The malformed values were `-1`, `tt0468569`, and `N/A`.

The pipeline kept the first occurrence of each eligible ID. That left
117 distinct IDs to request from TMDb. The six duplicate rows added no
new IDs. The four unusable rows reduced coverage of the supplied list.

Reconciliation: 127 = 1 blank + 3 invalid + 6 duplicates + 117 ready to fetch.

## 2. TMDb response structure

I inspected the cached details and credits for The Dark Knight (TMDb ID 155).
The goal was to see how TMDb represents movie attributes, collection
membership, genres, cast, and crew before deciding how to flatten them.

In [90]:
import json
import sqlite3
from contextlib import closing

cache_path = project_root / "data" / "cache" / "tmdb.sqlite"
movie_id = "155"

if not cache_path.is_file():
    raise FileNotFoundError(f"Run pipeline/extract.py first. Cache missing at {cache_path}")

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute(
        """
        SELECT endpoint, payload
        FROM responses
        WHERE movie_id = ?
        """,
        (movie_id,),
    ).fetchall()

responses = {endpoint: json.loads(payload) for endpoint, payload in rows}
assert responses.keys() == {"details", "credits"}, f"Missing response for {movie_id}"

details = responses["details"]
credits = responses["credits"]
assert str(details["id"]) == str(credits["id"]) == movie_id

print(f"Movie: {details['title']} ({movie_id})")
print("Details fields:", list(details))
print("Credits fields:", list(credits))

Movie: The Dark Knight (155)
Details fields: ['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
Credits fields: ['id', 'cast', 'crew']


In [91]:
collection = details["belongs_to_collection"]
first_billed_actor = min(credits["cast"], key=lambda actor: actor["order"])
directors = [person for person in credits["crew"] if person["job"] == "Director"]

print("Collection:", collection["id"], collection["name"])
print("Genres:", [genre["name"] for genre in details["genres"]])
print("First-billed actor:", first_billed_actor["name"])
print("First crew entry:", credits["crew"][0]["name"], credits["crew"][0]["job"])
print("Directors:", [person["name"] for person in directors])

Collection: 263 The Dark Knight Collection
Genres: ['Action', 'Thriller', 'Crime']
First-billed actor: Christian Bale
First crew entry: Jonathan Nolan Screenplay
Directors: ['Christopher Nolan']


### Finding and decision

The details response contains single movie values such as title and release
date, a nested collection object, and a list of genres. The credits response
contains separate cast and crew lists. Both responses share the same movie ID.

The main movie table will have one row per movie. Collection ID and name can
be extracted into that row. If we need genre-level analysis, each movie–genre
pair can go into a separate table so a movie with three genres is counted once
in movie-level totals.

For credits, billing order identifies the first-billed actor in this example.
The crew list includes several jobs, so directors are identified using the
Director job label. This example shows the structure. Coverage across the full
candidate list is checked in the next section.

## 3. Acquisition coverage

I compared the validated ID list with the fetch audit to account for every
requested movie. I checked movie details and credits separately because
a movie with details can still be used in analyses that do not need credits.

In [92]:
validated_path = project_root / "data" / "validated" / "validated_ids.csv"
fetch_audit_path = project_root / "data" / "cache" / "fetch_audit.csv"

validated = pd.read_csv(validated_path, dtype={"tmdb_id": str})
fetch_audit = pd.read_csv(fetch_audit_path, dtype={"tmdb_id": str})

assert validated["tmdb_id"].is_unique, "Duplicate IDs in the validated list."
assert fetch_audit["tmdb_id"].is_unique, "Duplicate IDs in the fetch audit."
assert set(validated["tmdb_id"]) == set(fetch_audit["tmdb_id"]), (
    "The fetch audit and validated list contain different IDs."
)

details_ok = fetch_audit["details"].eq("ok")
credits_ok = fetch_audit["credits"].eq("ok")

coverage = pd.Series({
    "Both responses available": (details_ok & credits_ok).sum(),
    "Details only": (details_ok & ~credits_ok).sum(),
    "Credits only": (~details_ok & credits_ok).sum(),
    "Neither response available": (~details_ok & ~credits_ok).sum(),
}, name="movies")

assert coverage.sum() == len(validated)
display(coverage.to_frame())

print(f"Movies with details available: {details_ok.sum()}/{len(validated)}")

,movies
Both responses available,112
Details only,0
Credits only,0
Neither response available,5


Movies with details available: 112/117


In [93]:
#incomplete movies
incomplete = fetch_audit.loc[
    ~(details_ok & credits_ok),
    ["tmdb_id", "details", "credits"],
]

display(incomplete)

,tmdb_id,details,credits
16,8471293,8471293 details: HTTP 404,8471293 credits: HTTP 404
32,9412887,9412887 details: HTTP 404,9412887 credits: HTTP 404
67,8471294,8471294 details: HTTP 404,8471294 credits: HTTP 404
85,9412888,9412888 details: HTTP 404,9412888 credits: HTTP 404
89,7733901,7733901 details: HTTP 404,7733901 credits: HTTP 404


### Finding and decision

Of the 117 validated movie IDs, 112 had both details and credits.
Five IDs had at least one unsuccessful request. The table above records
which endpoint failed for each ID.

Movies with available details can proceed to movie-level profiling. Movies
with missing credits are excluded from analyses that require director or
lead-actor information. IDs with missing details are excluded from analyses
that require movie attributes.

## 4. Movie details quality

I profiled the cached details for every candidate with a successful details
request. I checked for missing or zero financial values and unusable release
dates before deciding which movies were eligible for each analysis.

In [94]:
expected_ids = set(fetch_audit.loc[details_ok, "tmdb_id"])

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute("""
        SELECT movie_id, payload
        FROM responses
        WHERE endpoint = 'details'
    """).fetchall()

records = [
    {"tmdb_id": movie_id, **json.loads(payload)}
    for movie_id, payload in rows
    if movie_id in expected_ids
]

movies = pd.DataFrame(records)

assert movies["tmdb_id"].is_unique
assert set(movies["tmdb_id"]) == expected_ids, "Cache and fetch audit disagree."

print(f"Movies profiled: {len(movies)}")
display(movies[["tmdb_id", "title", "release_date", "budget", "revenue"]].head())

Movies profiled: 112


,tmdb_id,title,release_date,budget,revenue
0,155,The Dark Knight,2008-07-16,185000000,1008480529
1,165,Back to the Future Part II,1989-11-22,40000000,332861197
2,242,The Godfather Part III,1990-12-25,54000000,136766062
3,49026,The Dark Knight Rises,2012-07-17,250000000,1081041287
4,8077,Alien³,1992-05-22,50000000,159773545


In [95]:
budget = pd.to_numeric(movies["budget"], errors="coerce")
revenue = pd.to_numeric(movies["revenue"], errors="coerce")
release_date = pd.to_datetime(
    movies["release_date"], format="%Y-%m-%d", errors="coerce"
)

checks = pd.DataFrame({
    "Budget missing or non-numeric": budget.isna(),
    "Budget is zero": budget.eq(0),
    "Revenue missing or non-numeric": revenue.isna(),
    "Revenue is zero": revenue.eq(0),
    "Release date missing or invalid": release_date.isna(),
})

display(checks.sum().rename("movies_affected").to_frame())

affected = checks.any(axis=1)
print(f"Distinct movies affected: {affected.sum()}")
display(
    movies.loc[
        affected, ["tmdb_id", "title", "release_date", "budget", "revenue"]
    ].join(checks.loc[affected])
)

,movies_affected
Budget missing or non-numeric,0
Budget is zero,0
Revenue missing or non-numeric,0
Revenue is zero,0
Release date missing or invalid,0


Distinct movies affected: 0


,tmdb_id,title,release_date,budget,revenue,Budget missing or non-numeric,Budget is zero,Revenue missing or non-numeric,Revenue is zero,Release date missing or invalid


### Finding

All movies with cached details had parseable release dates and numeric,
nonzero budgets and revenues. These five checks flagged zero movies.
I then checked for negative amounts and reviewed the highest and lowest
revenue-to-budget ratios.

In [96]:
print(f"Negative budgets: {budget.lt(0).sum()}")
print(f"Negative revenues: {revenue.lt(0).sum()}")

positive_amounts = budget.gt(0) & revenue.gt(0)
ratios = movies.loc[positive_amounts, ["tmdb_id", "title"]].copy()
ratios["budget"] = budget.loc[positive_amounts]
ratios["revenue"] = revenue.loc[positive_amounts]
ratios["revenue_to_budget"] = ratios["revenue"] / ratios["budget"]

print("\nFive highest ratios")
display(ratios.nlargest(5, "revenue_to_budget"))

print("\nFive lowest ratios")
display(ratios.nsmallest(5, "revenue_to_budget"))

Negative budgets: 0
Negative revenues: 0

Five highest ratios


,tmdb_id,title,budget,revenue,revenue_to_budget
23,11,Star Wars,11000000,775398007,70.490728
85,578,Jaws,7000000,470653000,67.236143
59,238,The Godfather,6000000,245066411,40.844402
67,429,"The Good, the Bad and the Ugly",1200000,38900000,32.416667
49,1891,The Empire Strikes Back,18000000,538400000,29.911111



Five lowest ratios


,tmdb_id,title,budget,revenue,revenue_to_budget
24,278,The Shawshank Redemption,25000000,28341469,1.133659
56,78,Blade Runner,28000000,41722424,1.490087
71,550,Fight Club,63000000,100853753,1.600853
55,641,Requiem for a Dream,4500000,7390108,1.642246
102,534,Terminator Salvation,200000000,371353001,1.856765


In [97]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pipeline.transform import flatten_movie_details

flat_movies = flatten_movie_details(movies)

assert flat_movies["tmdb_id"].is_unique
assert flat_movies["tmdb_id"].equals(movies["tmdb_id"])

print(f"Movies before flattening: {len(movies)}")
print(f"Movies after flattening:  {len(flat_movies)}")

display(
    flat_movies.loc[
        flat_movies["tmdb_id"].isin(["155", "278"]),
        ["tmdb_id", "title", "collection_id", "collection_name", "revenue_to_budget"],
    ]
)

Movies before flattening: 112
Movies after flattening:  112


,tmdb_id,title,collection_id,collection_name,revenue_to_budget
0,155,The Dark Knight,263,The Dark Knight Collection,5.451246
24,278,The Shawshank Redemption,<NA>,NaN,1.133659


## Mean Versus Median

Compare the two summary measures before choosing how to describe the typical
film and before the franchise, standalone, and talent rankings. Reuse `flat_movies`,
the original 112-film baseline created immediately above. This comparison excludes
hypothetical missing films and repeated input rows. A positive gap means the
franchise group has the higher value. Calculations use full precision; displayed
ratios are rounded to three decimals.


In [98]:
import pandas as pd

mean_median_films = flat_movies[
    ["tmdb_id", "release_date", "collection_id", "revenue_to_budget"]
].dropna(subset=["revenue_to_budget"]).copy()

assert mean_median_films["tmdb_id"].is_unique, "Expected one row per film."

mean_median_films["group"] = mean_median_films["collection_id"].notna().map({
    True: "Franchise",
    False: "Standalone",
})
mean_median_release_dates = pd.to_datetime(mean_median_films["release_date"], errors="coerce")
mean_median_all_years = mean_median_films.assign(window="All years")
mean_median_recent = mean_median_films.loc[
    mean_median_release_dates.ge(pd.Timestamp("2000-01-01"))
].assign(window="2000 onward")

mean_median_summary = (
    pd.concat([mean_median_all_years, mean_median_recent], ignore_index=True)
    .groupby(["window", "group"], sort=False)["revenue_to_budget"]
    .agg(films="size", mean_ratio="mean", median_ratio="median")
)

assert mean_median_summary["films"].to_dict() == {
    ("All years", "Franchise"): 96,
    ("All years", "Standalone"): 16,
    ("2000 onward", "Franchise"): 63,
    ("2000 onward", "Standalone"): 8,
}, "Film counts differ from the memo baseline."

display(mean_median_summary.round(3))

mean_median_gaps = (
    mean_median_summary.xs("Franchise", level="group")[["mean_ratio", "median_ratio"]]
    - mean_median_summary.xs("Standalone", level="group")[["mean_ratio", "median_ratio"]]
).rename(columns={"mean_ratio": "mean_gap", "median_ratio": "median_gap"})

print("Gap = franchise minus standalone")
display(mean_median_gaps.round(3))


films  mean_ratio  median_ratio
window      group                                      
All years   Franchise      96       8.540         5.624
            Standalone     16       8.284         3.845
2000 onward Franchise      63       5.411         5.052
            Standalone      8       4.901         3.845

Gap = franchise minus standalone


,mean_gap,median_gap
window,,
All years,0.256,1.779
2000 onward,0.510,1.207


### Choice of summary measure

The mean exceeds the median in every group and window, consistent with the
high ratios reviewed above pulling the arithmetic average upward. For describing
a typical film, use the **median** as the main measure because it is less affected
by unusually large ratios. Keep the mean as a sensitivity comparison, rather
than choose the statistic according to which recommendation it supports.

Franchises lead with either measure in both windows. However, the all-years gap
is **0.256 using means**, compared with **1.779 using medians**; from 2000 onward,
the gaps are **0.510** and **1.207**. Thus the leader is unchanged, but the size of
the apparent advantage depends on the summary measure. Neither statistic measures
net profit or expected streaming returns. This baseline comparison does not replace
the missing-record and duplicate-row scenarios later in the notebook.


In [99]:
title_check = flat_movies[["tmdb_id", "title", "release_date"]].copy()

title_check["title_key"] = (
    title_check["title"]
    .astype("string")
    .str.normalize("NFKC")
    .str.casefold()
    .str.replace(r"[^\w]+", " ", regex=True)
    .str.strip()
)
title_check["release_year"] = title_check["release_date"].dt.year

# A film needs both a title and a release year for this check.
eligible = title_check.dropna(subset=["title_key", "release_year"])
eligible = eligible.loc[eligible["title_key"].ne("")].copy()

keys = ["title_key", "release_year"]
distinct_ids = eligible.groupby(keys)["tmdb_id"].transform("nunique")

possible_duplicates = eligible.loc[distinct_ids.gt(1)].sort_values(
    [*keys, "tmdb_id"]
)

print(f"Films eligible for the check: {len(eligible)} of {len(title_check)}")
print(f"Possible duplicate groups: {len(possible_duplicates[keys].drop_duplicates())}")
display(possible_duplicates[["tmdb_id", "title", "release_date"]])

Films eligible for the check: 112 of 112
Possible duplicate groups: 0


,tmdb_id,title,release_date


In [100]:
imdb_check = movies.loc[
    movies["tmdb_id"].astype(str).eq("155"),
    ["tmdb_id", "title", "imdb_id"],
]

display(imdb_check)

,tmdb_id,title,imdb_id
0,155,The Dark Knight,tt0468569


In [101]:
#franchise vs standalone movies
profile = flat_movies.assign(
    movie_group=flat_movies["collection_id"].notna().map({
        True: "Franchise",
        False: "Standalone",
    })
)

windows = {
    "All years": profile,
    "2000 onward": profile.loc[profile["release_date"].dt.year.ge(2000)],
}

coverage = pd.concat({
    window: films.groupby("movie_group").agg(
        films=("tmdb_id", "size"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
    )
    for window, films in windows.items()
}).rename_axis(["time_window", "movie_group"])

display(coverage)

films  films_with_ratio  median_ratio
time_window movie_group                                       
All years   Franchise       96                96      5.623799
            Standalone      16                16      3.844812
2000 onward Franchise       63                63      5.051939
            Standalone       8                 8      3.844812

Franchise films had a higher median revenue-to-budget ratio in both time windows. The gap narrowed from 1.78 to 1.21 after restricting the comparison to films released from 2000 onward. The later comparison includes 63 franchise films and 8 standalones, so its standalone estimate is based on a small sample.

In [102]:
comparison = []

for window, films in windows.items():
    franchise_films = films.loc[films["collection_id"].notna()]
    standalone_films = films.loc[films["collection_id"].isna()]

    collections = (
        franchise_films.groupby("collection_id")
        .agg(
            collection_name=("collection_name", "first"),
            films_in_sample=("tmdb_id", "size"),
            median_ratio=("revenue_to_budget", "median"),
        )
        .sort_values("films_in_sample", ascending=False)
    )

    print(window)
    display(collections)

    comparison.append({
        "time_window": window,
        "collections_with_ratio": collections["median_ratio"].count(),
        "franchise_median_equal_weight": collections["median_ratio"].median(),
        "standalone_median": standalone_films["revenue_to_budget"].median(),
    })

display(pd.DataFrame(comparison).set_index("time_window").round(2))

All years


,collection_name,films_in_sample,median_ratio
collection_id,,,
1241,Harry Potter Collection,8,6.165086
10,Star Wars Collection,6,13.401690
295,Pirates of the Caribbean Collection,5,3.460532
87359,Mission: Impossible Collection,5,4.551444
2150,Shrek Collection,4,5.634445
1570,Die Hard Collection,4,3.777223
528,The Terminator Collection,4,3.625805
84,Indiana Jones Collection,4,10.887634
10194,Toy Story Collection,4,5.831315


2000 onward


,collection_name,films_in_sample,median_ratio
collection_id,,,
1241,Harry Potter Collection,8,6.165086
295,Pirates of the Caribbean Collection,5,3.460532
2150,Shrek Collection,4,5.634445
87359,Mission: Impossible Collection,4,4.461275
86311,The Avengers Collection,4,6.872545
263,The Dark Knight Collection,3,4.324165
304,Ocean's Collection,3,3.662501
119,The Lord of the Rings Collection,3,11.725157
121938,The Hobbit Collection,3,3.833600


,collections_with_ratio,franchise_median_equal_weight,standalone_median
time_window,,,
All years,32,5.73,3.84
2000 onward,24,4.39,3.84


In [103]:
import importlib
import pipeline.transform

importlib.reload(pipeline.transform)
from pipeline.transform import flatten_lead_actors

with closing(sqlite3.connect(cache_path)) as connection:
    credit_rows = connection.execute("""
        SELECT movie_id, payload
        FROM responses
        WHERE endpoint = 'credits'
    """).fetchall()

movie_ids = set(flat_movies["tmdb_id"])
credits_by_movie = {
    movie_id: json.loads(payload)
    for movie_id, payload in credit_rows
    if movie_id in movie_ids
}

lead_actors = flatten_lead_actors(credits_by_movie)

movies_with_leads = flat_movies.merge(
    lead_actors, on="tmdb_id", how="left", validate="one_to_one"
)

assert len(movies_with_leads) == len(flat_movies)

print(f"Movies with a first-billed actor: {len(lead_actors)}/{len(flat_movies)}")
display(
    movies_with_leads.loc[
        movies_with_leads["tmdb_id"].eq("155"),
        ["tmdb_id", "title", "lead_actor_name"],
    ]
)

Movies with a first-billed actor: 112/112


,tmdb_id,title,lead_actor_name
0,155,The Dark Knight,Christian Bale


In [104]:
importlib.reload(pipeline.transform)

directors = pipeline.transform.flatten_directors(credits_by_movie)
director_counts = directors.groupby("tmdb_id").size()

print(f"Movies with a director: {len(director_counts)}/{len(flat_movies)}")
print(f"Movies with more than one director: {director_counts.gt(1).sum()}")
print(f"Movie–director pairs: {len(directors)}")

display(directors.loc[directors["tmdb_id"].eq("155")])

Movies with a director: 112/112
Movies with more than one director: 9
Movie–director pairs: 122


,tmdb_id,director_id,director_name
0,155,525,Christopher Nolan


In [105]:
director_films = directors.merge(
    flat_movies[["tmdb_id", "title", "release_date", "revenue_to_budget"]],
    on="tmdb_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert len(director_films) == len(directors)
assert director_films["_merge"].eq("both").all(), "Some director rows have no matching movie."

director_films = director_films.drop(columns="_merge")

display(
    director_films.loc[
        director_films["director_id"].eq(525),
        ["director_name", "title", "revenue_to_budget"],
    ]
)

,director_name,title,revenue_to_budget
0,Christopher Nolan,The Dark Knight,5.451246
3,Christopher Nolan,The Dark Knight Rises,4.324165
40,Christopher Nolan,Inception,5.243941
54,Christopher Nolan,The Prestige,2.741908
81,Christopher Nolan,Memento,4.451123
114,Christopher Nolan,Batman Begins,2.494791
118,Christopher Nolan,Interstellar,4.524889


In [106]:
director_summary = (
    director_films.groupby("director_id", as_index=False)
    .agg(
        director_name=("director_name", "first"),
        films=("tmdb_id", "nunique"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
)

min_films = 3

director_ranking = (
    director_summary.loc[director_summary["films_with_ratio"].ge(min_films)]
    .sort_values(
        ["median_ratio", "films_with_ratio", "director_id"],
        ascending=[False, False, True],
    )
)

print(f"Directors with at least {min_films} usable film ratios: {len(director_ranking)}")
display(director_ranking.head(10).round(2))

Directors with at least 3 usable film ratios: 13


,director_id,director_name,films,films_with_ratio,median_ratio,lowest_ratio
6,488,Steven Spielberg,8,8,13.25,4.25
3,24,Robert Zemeckis,4,4,10.32,6.13
19,2710,James Cameron,3,3,9.91,5.08
0,1,George Lucas,4,4,8.31,5.41
16,1776,Francis Ford Coppola,3,3,7.89,2.53
4,108,Peter Jackson,6,6,6.73,3.82
5,138,Quentin Tarantino,3,3,6.03,5.07
40,11343,David Yates,4,4,5.04,3.74
34,9339,Lilly Wachowski,3,3,4.92,2.83
35,9340,Lana Wachowski,3,3,4.92,2.83


In [107]:
for cutoff in (3, 2):
    eligible = director_summary.loc[
        director_summary["films_with_ratio"].ge(cutoff)
    ].sort_values(
        ["median_ratio", "films_with_ratio", "director_id"],
        ascending=[False, False, True],
    )

    print(f"Minimum {cutoff} films: {len(eligible)} eligible directors")
    display(
        eligible[["director_name", "films_with_ratio", "median_ratio", "lowest_ratio"]]
        .head(5)
        .round(2)
    )

Minimum 3 films: 13 eligible directors


,director_name,films_with_ratio,median_ratio,lowest_ratio
6,Steven Spielberg,8,13.25,4.25
3,Robert Zemeckis,4,10.32,6.13
19,James Cameron,3,9.91,5.08
0,George Lucas,4,8.31,5.41
16,Francis Ford Coppola,3,7.89,2.53


Minimum 2 films: 28 eligible directors


,director_name,films_with_ratio,median_ratio,lowest_ratio
6,Steven Spielberg,8,13.25,4.25
3,Robert Zemeckis,4,10.32,6.13
19,James Cameron,3,9.91,5.08
31,John Lasseter,2,9.45,5.53
26,Carlos Saldanha,2,9.06,8.26


In [108]:
actor_summary = (
    movies_with_leads.groupby("lead_actor_id", as_index=False)
    .agg(
        lead_actor_name=("lead_actor_name", "first"),
        films=("tmdb_id", "nunique"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
)

for cutoff in (3, 2):
    actor_ranking = (
        actor_summary.loc[actor_summary["films_with_ratio"].ge(cutoff)]
        .sort_values(
            ["median_ratio", "films_with_ratio", "lead_actor_id"],
            ascending=[False, False, True],
        )
    )

    print(f"Minimum {cutoff} films: {len(actor_ranking)} eligible actors")
    display(actor_ranking.head(5).round(2))

Minimum 3 films: 19 eligible actors


,lead_actor_id,lead_actor_name,films,films_with_ratio,median_ratio,lowest_ratio
0,2,Mark Hamill,3,3,29.91,17.70
5,109,Elijah Wood,3,3,11.73,9.37
1,3,Harrison Ford,5,5,9.88,1.49
11,521,Michael J. Fox,3,3,8.32,6.13
44,15757,Ray Romano,3,3,8.26,6.50


Minimum 2 films: 25 eligible actors


,lead_actor_id,lead_actor_name,films,films_with_ratio,median_ratio,lowest_ratio
0,2,Mark Hamill,3,3,29.91,17.70
27,3896,Liam Neeson,2,2,11.85,9.10
5,109,Elijah Wood,3,3,11.73,9.37
30,4783,Sam Neill,2,2,10.38,3.97
1,3,Harrison Ford,5,5,9.88,1.49


## Talent rankings — 2000 onward

These rankings reuse `movies_with_leads` and `director_films` from the cached
movie and credit data above. Filter to release dates on or after **2000-01-01**
and usable revenue-to-budget ratios **before** grouping. Each person needs at
least **3 distinct films** in this window. Co-directed films count once for each
director. The ranking matches the all-years rules: median ratio descending,
usable film count descending, then person ID ascending. Only display values are
rounded; ranking uses full precision. Show up to three qualifying people per role.


In [109]:
talent_cutoff_2000 = pd.Timestamp("2000-01-01")
talent_min_films_2000 = 3

actor_films_2000 = movies_with_leads.loc[
    movies_with_leads["release_date"].ge(talent_cutoff_2000)
    & movies_with_leads["revenue_to_budget"].notna()
].drop_duplicates(subset=["tmdb_id", "lead_actor_id"])

actor_summary_2000 = (
    actor_films_2000.groupby("lead_actor_id", as_index=False)
    .agg(
        lead_actor_name=("lead_actor_name", "first"),
        films=("tmdb_id", "nunique"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
)
assert actor_summary_2000["films"].eq(actor_summary_2000["films_with_ratio"]).all()
actor_ranking_2000 = (
    actor_summary_2000.loc[actor_summary_2000["films_with_ratio"].ge(talent_min_films_2000)]
    .sort_values(
        ["median_ratio", "films_with_ratio", "lead_actor_id"],
        ascending=[False, False, True],
    )
)
actors_top3_2000 = actor_ranking_2000[
    ["lead_actor_name", "films_with_ratio", "median_ratio", "lowest_ratio"]
].head(3).rename(columns={"lead_actor_name": "Actor", "films_with_ratio": "Film count"}).reset_index(drop=True)

print(f"Actors qualifying with at least {talent_min_films_2000} distinct usable films from 2000 onward: {len(actor_ranking_2000)}")
display(actors_top3_2000.round({"median_ratio": 2, "lowest_ratio": 2}))


Actors qualifying with at least 3 distinct usable films from 2000 onward: 10


,Actor,Film count,median_ratio,lowest_ratio
0,Elijah Wood,3,11.73,9.37
1,Ray Romano,3,8.26,6.50
2,Daniel Radcliffe,8,6.17,3.74


In [110]:
director_films_2000 = director_films.loc[
    director_films["release_date"].ge(talent_cutoff_2000)
    & director_films["revenue_to_budget"].notna()
].drop_duplicates(subset=["tmdb_id", "director_id"])

director_summary_2000 = (
    director_films_2000.groupby("director_id", as_index=False)
    .agg(
        director_name=("director_name", "first"),
        films=("tmdb_id", "nunique"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
)
assert director_summary_2000["films"].eq(director_summary_2000["films_with_ratio"]).all()
director_ranking_2000 = (
    director_summary_2000.loc[director_summary_2000["films_with_ratio"].ge(talent_min_films_2000)]
    .sort_values(
        ["median_ratio", "films_with_ratio", "director_id"],
        ascending=[False, False, True],
    )
)
directors_top3_2000 = director_ranking_2000[
    ["director_name", "films_with_ratio", "median_ratio", "lowest_ratio"]
].head(3).rename(columns={"director_name": "Director", "films_with_ratio": "Film count"}).reset_index(drop=True)

print(f"Directors qualifying with at least {talent_min_films_2000} distinct usable films from 2000 onward: {len(director_ranking_2000)}")
display(directors_top3_2000.round({"median_ratio": 2, "lowest_ratio": 2}))


Directors qualifying with at least 3 distinct usable films from 2000 onward: 5


,Director,Film count,median_ratio,lowest_ratio
0,Peter Jackson,6,6.73,3.82
1,David Yates,4,5.04,3.74
2,Gore Verbinski,3,4.68,3.21


In [111]:
franchise_summary = (
    flat_movies.loc[flat_movies["collection_id"].notna()]
    .groupby("collection_id", as_index=False)
    .agg(
        collection_name=("collection_name", "first"),
        films_in_sample=("tmdb_id", "nunique"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
)

franchise_shortlist = (
    franchise_summary.loc[franchise_summary["films_with_ratio"].ge(3)]
    .sort_values(
        ["median_ratio", "films_with_ratio", "collection_id"],
        ascending=[False, False, True],
    )
    .head(3)
)

display(franchise_shortlist.round(2))

,collection_id,collection_name,films_in_sample,films_with_ratio,median_ratio,lowest_ratio
0,10,Star Wars Collection,6,6,13.40,5.41
2,119,The Lord of the Rings Collection,3,3,11.73,9.37
1,84,Indiana Jones Collection,4,4,10.89,4.25


In [112]:
standalone_shortlist = (
    flat_movies.loc[
        flat_movies["collection_id"].isna()
        & flat_movies["revenue_to_budget"].notna(),
        [
            "tmdb_id", "title", "release_date", "budget", "revenue",
            "revenue_to_budget", "vote_average", "vote_count",
        ],
    ]
    .sort_values(
        ["revenue_to_budget", "revenue", "tmdb_id"],
        ascending=[False, False, True],
    )
    .head(5)
)

display(standalone_shortlist.round({"revenue_to_budget": 2}))

,tmdb_id,title,release_date,budget,revenue,revenue_to_budget,vote_average,vote_count
67,429,"The Good, the Bad and the Ugly",1966-12-23,1200000,38900000,32.42,8.468,10165
6,680,Pulp Fiction,1994-09-10,8000000,213928762,26.74,8.480,31368
96,424,Schindler's List,1993-12-15,22000000,321365567,14.61,8.570,18206
63,129,Spirited Away,2001-07-20,19000000,274925095,14.47,8.537,19246
66,13,Forrest Gump,1994-06-23,55000000,677387716,12.32,8.462,30924


In [113]:
standalone_shortlist_2000 = (
    flat_movies.loc[
        flat_movies["collection_id"].isna()
        & flat_movies["release_date"].dt.year.ge(2000)
        & flat_movies["revenue_to_budget"].notna()
    ]
    .sort_values(
        ["revenue_to_budget", "revenue", "tmdb_id"],
        ascending=[False, False, True],
    )
    .head(5)
)

display(
    standalone_shortlist_2000[
        ["title", "release_date", "revenue_to_budget"]
    ].round(2)
)

C:\Users\User\AppData\Local\Temp\ipykernel_25304\1449995129.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,title,release_date,revenue_to_budget
63,Spirited Away,2001-07-20,14.47
37,Inception,2010-07-15,5.24
108,Interstellar,2014-11-04,4.52
75,Memento,2000-10-11,4.45
36,The Departed,2006-10-04,3.24


In [114]:
franchise_shortlist_2000 = (
    flat_movies.loc[
        flat_movies["collection_id"].notna()
        & flat_movies["release_date"].dt.year.ge(2000)
    ]
    .groupby("collection_id", as_index=False)
    .agg(
        collection_name=("collection_name", "first"),
        films_with_ratio=("revenue_to_budget", "count"),
        median_ratio=("revenue_to_budget", "median"),
        lowest_ratio=("revenue_to_budget", "min"),
    )
    .query("films_with_ratio >= 3")
    .sort_values(
        ["median_ratio", "films_with_ratio", "collection_id"],
        ascending=[False, False, True],
    )
    .head(3)
)

display(franchise_shortlist_2000.round(2))

,collection_id,collection_name,films_with_ratio,median_ratio,lowest_ratio
2,119,The Lord of the Rings Collection,3,11.73,9.37
13,8354,Ice Age Collection,3,8.26,6.50
17,86311,The Avengers Collection,4,6.87,5.98


In [115]:
deal_candidates = flat_movies.loc[
    flat_movies["collection_id"].isin([10, 119]),
    ["collection_name", "title", "release_date", "revenue_to_budget"],
].copy()

deal_candidates["released_2000_onward"] = deal_candidates["release_date"].dt.year.ge(2000)

display(
    deal_candidates.sort_values(["collection_name", "release_date"])
    .round({"revenue_to_budget": 2})
    .reset_index(drop=True)
)

,collection_name,title,release_date,revenue_to_budget,released_2000_onward
0,Star Wars Collection,Star Wars,1977-05-25,70.49,False
1,Star Wars Collection,The Empire Strikes Back,1980-05-20,29.91,False
2,Star Wars Collection,Return of the Jedi,1983-05-25,17.70,False
3,Star Wars Collection,Star Wars: Episode I - The Phantom Menace,1999-05-19,9.10,False
4,Star Wars Collection,Star Wars: Episode II - Attack of the Clones,2002-05-15,5.41,True
5,Star Wars Collection,Star Wars: Episode III - Revenge of the Sith,2005-05-17,7.52,True
6,The Lord of the Rings Collection,The Lord of the Rings: The Fellowship of the Ring,2001-12-18,9.37,True
7,The Lord of the Rings Collection,The Lord of the Rings: The Two Towers,2002-12-18,11.73,True
8,The Lord of the Rings Collection,The Lord of the Rings: The Return of the King,2003-12-17,11.90,True


## Additional decision notes

### Standalone shortlist

I ranked individual films by revenue-to-budget ratio and used higher
reported revenue to break ties. The all-years list includes four films
released before 2000. When I applied the 2000 cutoff, Spirited Away was
the only film that stayed in the top five, moving from fourth to first.
The cutoff therefore changes which standalone titles Kwame would pursue.

### Talent ranking

I required three films with usable ratios for the main ranking and tested
a two-film minimum. The lower cutoff increased the eligible pool from 13
to 28 directors and from 19 to 25 first-billed actors. The top three
directors stayed the same, and Mark Hamill remained the top actor.
John Lasseter, Carlos Saldanha, Liam Neeson, and Sam Neill entered their
respective top fives with two films each.

The three-film rule gives each main pick more evidence, but it can
exclude promising names. Several films may also come from one franchise,
so a strong ranking may reflect the franchise’s appeal as well as the
person’s contribution.

## Optional live TMDb spot-check

Live requests are disabled by default. The cached analysis and all subsequent
sections run without these checks. To perform a fresh spot-check intentionally,
set `RUN_LIVE_SPOT_CHECKS = True` in the next cell, configure your local API key,
and run both spot-check cells. Their historical live results are not needed to
reproduce the cached recommendations.


In [127]:
RUN_LIVE_SPOT_CHECKS = False

if not RUN_LIVE_SPOT_CHECKS:
    print("Live movie-detail spot-check skipped; cached analysis remains available.")
else:
    import os

    import requests
    from dotenv import load_dotenv

    load_dotenv(project_root / ".env")
    api_key = os.getenv("TMDB_API_KEY")
    if not api_key:
        raise ValueError("TMDB_API_KEY is missing from .env")

    sample_ids = [
        "155",  # The Dark Knight: known collection
        "11",   # Star Wars: unusually high ratio
        "129",  # Spirited Away: top standalone from 2000 onward
        str(standalone_shortlist_2000.iloc[-1]["tmdb_id"]),  # Fifth-place cutoff
    ]

    checks = []

    for movie_id in sample_ids:
        cached = movies.loc[movies["tmdb_id"].eq(movie_id)].iloc[0]

        try:
            response = requests.get(
                f"https://api.themoviedb.org/3/movie/{movie_id}",
                params={"api_key": api_key},
                timeout=20,
            )
        except requests.RequestException:
            checks.append({"tmdb_id": movie_id, "title": cached["title"], "result": "Request failed"})
            continue

        if response.status_code != 200:
            checks.append({
                "tmdb_id": movie_id,
                "title": cached["title"],
                "result": f"HTTP {response.status_code}",
            })
            continue

        live = response.json()
        fields = ["title", "budget", "revenue", "release_date", "belongs_to_collection"]
        changed = [field for field in fields if cached[field] != live.get(field)]

        checks.append({
            "tmdb_id": movie_id,
            "title": cached["title"],
            "result": "Match" if not changed else ", ".join(changed),
        })

    print("Checked on", pd.Timestamp.now(tz="UTC").date(), "UTC")
    display(pd.DataFrame(checks))

Live movie-detail spot-check skipped; cached analysis remains available.


In [128]:
if not globals().get("RUN_LIVE_SPOT_CHECKS", False):
    print("Live credits spot-check skipped; cached analysis remains available.")
else:
    from pipeline.transform import flatten_directors, flatten_lead_actors

    credit_sample_ids = ["155", director_counts[director_counts.gt(1)].index[0]]
    credit_checks = []

    for movie_id in credit_sample_ids:
        try:
            response = requests.get(
                f"https://api.themoviedb.org/3/movie/{movie_id}/credits",
                params={"api_key": api_key},
                timeout=20,
            )
        except requests.RequestException:
            credit_checks.append({"tmdb_id": movie_id, "result": "Request failed"})
            continue

        if response.status_code != 200:
            credit_checks.append({"tmdb_id": movie_id, "result": f"HTTP {response.status_code}"})
            continue

        live_credits = {movie_id: response.json()}
        live_actors = flatten_lead_actors(live_credits)
        live_directors = flatten_directors(live_credits)

        cached_actor_ids = set(
            lead_actors.loc[lead_actors["tmdb_id"].eq(movie_id), "lead_actor_id"]
        )
        cached_director_ids = set(
            directors.loc[directors["tmdb_id"].eq(movie_id), "director_id"]
        )

        credit_checks.append({
            "tmdb_id": movie_id,
            "first_billed_actor_matches": cached_actor_ids == set(live_actors["lead_actor_id"]),
            "directors_match": cached_director_ids == set(live_directors["director_id"]),
            "directors_checked": len(cached_director_ids),
        })

    display(pd.DataFrame(credit_checks))

Live credits spot-check skipped; cached analysis remains available.


## Genre analysis

In [118]:
from contextlib import closing
import json
from pathlib import Path
import sqlite3

import pandas as pd

project_root = Path.cwd()
if project_root.name.lower() == "data profile":
    project_root = project_root.parent

cache_path = project_root / "data" / "cache" / "tmdb.sqlite"
if not cache_path.is_file():
    raise FileNotFoundError(f"Cache missing at {cache_path}")

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute(
        "SELECT movie_id, payload FROM responses WHERE endpoint = 'details'"
    ).fetchall()

movies = pd.DataFrame([
    {"tmdb_id": str(movie_id), **json.loads(payload)}
    for movie_id, payload in rows
])

if "genres" not in movies.columns:
    raise ValueError("The cached movie details do not contain a 'genres' column.")

movie_genres = (
    movies[["tmdb_id", "genres"]]
    .explode("genres", ignore_index=True)
    .dropna(subset=["genres"])
    .assign(
        genre_id=lambda rows: rows["genres"].map(lambda genre: genre["id"]),
        genre_name=lambda rows: rows["genres"].map(lambda genre: genre["name"]),
    )
    .drop(columns="genres")
    .drop_duplicates(subset=["tmdb_id", "genre_id"])
    .reset_index(drop=True)
)

print(f"Movies with at least one genre: {movie_genres['tmdb_id'].nunique()}/{len(movies)}")
print(f"Distinct genres: {movie_genres['genre_id'].nunique()}")

display(movie_genres.loc[movie_genres["tmdb_id"].eq("155")])

Movies with at least one genre: 112/112
Distinct genres: 16


,tmdb_id,genre_id,genre_name
0,155,28,Action
1,155,53,Thriller
2,155,80,Crime


In [119]:
# Keep only IDs from the current run. SQLite may contain older movies too.
validated_path = project_root / "data" / "validated" / "validated_ids.csv"
validated_ids = pd.read_csv(validated_path, dtype={"tmdb_id": str})

genre_details = movies.loc[
    movies["tmdb_id"].isin(validated_ids["tmdb_id"]),
    ["tmdb_id", "release_date", "budget", "revenue"],
].copy()

genre_details["release_date"] = pd.to_datetime(
    genre_details["release_date"], format="%Y-%m-%d", errors="coerce"
)
budget = pd.to_numeric(genre_details["budget"], errors="coerce")
revenue = pd.to_numeric(genre_details["revenue"], errors="coerce")

genre_details["reported_revenue"] = revenue.where(revenue.gt(0))
genre_details["revenue_to_budget"] = (
    genre_details["reported_revenue"] / budget.where(budget.gt(0))
)

genre_films = movie_genres.merge(
    genre_details[
        ["tmdb_id", "release_date", "reported_revenue", "revenue_to_budget"]
    ],
    on="tmdb_id",
    how="inner",
    validate="many_to_one",
)

print(f"Films in the genre analysis: {genre_films['tmdb_id'].nunique()}")
display(genre_films.loc[genre_films["tmdb_id"].eq("155")])

Films in the genre analysis: 112


,tmdb_id,genre_id,genre_name,release_date,reported_revenue,revenue_to_budget
0,155,28,Action,2008-07-16,1008480529,5.451246
1,155,53,Thriller,2008-07-16,1008480529,5.451246
2,155,80,Crime,2008-07-16,1008480529,5.451246


In [120]:
release_year = genre_films["release_date"].dt.year
genre_windows = {
    "All years": genre_films,
    "Before 2000": genre_films.loc[release_year.lt(2000)],
    "2000 onward": genre_films.loc[release_year.ge(2000)],
}

genre_results = {}

for period, films in genre_windows.items():
    summary = (
        films.groupby("genre_id", as_index=False)
        .agg(
            genre=("genre_name", "first"),
            films_in_sample=("tmdb_id", "nunique"),
            films_with_revenue=("reported_revenue", "count"),
            median_revenue=("reported_revenue", "median"),
            films_with_ratio=("revenue_to_budget", "count"),
            median_ratio=("revenue_to_budget", "median"),
        )
        .sort_values(
            ["median_revenue", "films_with_revenue", "genre_id"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    eligible = summary["films_with_revenue"].ge(5)
    summary["revenue_rank"] = (
        eligible.cumsum().where(eligible).astype("Int64")
    )
    summary["median_revenue_millions"] = summary["median_revenue"] / 1_000_000
    genre_results[period] = summary

    print(period)
    display(summary[
        [
            "revenue_rank", "genre", "films_in_sample", "films_with_revenue",
            "median_revenue_millions", "films_with_ratio", "median_ratio",
        ]
    ].round(2))

All years


,revenue_rank,genre,films_in_sample,films_with_revenue,median_revenue_millions,films_with_ratio,median_ratio
0,1,Fantasy,27,27,926.29,27,5.33
1,<NA>,Romance,2,2,804.97,2,9.27
2,2,Adventure,60,60,781.02,60,6.13
3,3,Animation,17,17,650.00,17,6.22
4,4,Family,17,17,650.00,17,6.22
5,5,Action,61,61,585.17,61,5.03
6,6,Comedy,19,19,563.75,19,6.50
7,7,Science Fiction,36,36,529.86,36,6.03
8,8,Thriller,33,33,366.10,33,4.32
9,<NA>,History,1,1,321.37,1,14.61


Before 2000


,revenue_rank,genre,films_in_sample,films_with_revenue,median_revenue_millions,films_with_ratio,median_ratio
0,<NA>,Romance,1,1,677.39,1,12.32
1,<NA>,Animation,3,3,497.38,3,13.37
2,<NA>,Family,3,3,497.38,3,13.37
3,1,Adventure,15,15,470.65,15,13.37
4,2,Science Fiction,17,17,391.55,17,9.10
5,3,Action,19,19,389.93,19,8.47
6,4,Comedy,8,8,362.20,8,11.10
7,<NA>,History,1,1,321.37,1,14.61
8,<NA>,War,1,1,321.37,1,14.61
9,<NA>,Fantasy,1,1,296.64,1,9.89


2000 onward


,revenue_rank,genre,films_in_sample,films_with_revenue,median_revenue_millions,films_with_ratio,median_ratio
0,<NA>,Romance,1,1,932.54,1,6.22
1,1,Fantasy,26,26,929.42,26,5.23
2,2,Adventure,45,45,871.37,45,5.33
3,3,Comedy,11,11,752.60,11,6.14
4,4,Action,42,42,716.66,42,4.46
5,5,Animation,14,14,655.50,14,6.18
6,6,Family,14,14,655.50,14,6.18
7,7,Science Fiction,19,19,649.40,19,4.52
8,8,Thriller,20,20,391.01,20,3.91
9,9,Crime,10,10,368.48,10,4.70


Fantasy had the highest median reported box-office revenue across all years ($926.29 million across 27 films) and from 2000 onward ($929.42 million across 26 films). Adventure led before 2000 ($470.65 million across 15 films) and placed second from 2000 onward. Fantasy had only one pre-2000 film, so its overall result largely reflects newer releases. Among films released from 2000 onward, Animation earned more per dollar of production budget at the median than Fantasy (6.18× compared with 5.23×). These results describe historical box-office performance within the candidate list. Kwame can use them to decide which genres to explore, while evaluating each title on its own licensing terms and expected streaming demand.

In [121]:
animation_ids = set(
    genre_films.loc[genre_films["genre_name"].eq("Animation"), "tmdb_id"]
)
family_ids = set(
    genre_films.loc[genre_films["genre_name"].eq("Family"), "tmdb_id"]
)

print("Films in both genres:", len(animation_ids & family_ids))
print("Films only in Animation:", len(animation_ids - family_ids))
print("Films only in Family:", len(family_ids - animation_ids))

Films in both genres: 17
Films only in Animation: 0
Films only in Family: 0


All 17 films tagged Animation were also tagged Family, and every Family film was tagged Animation. The two genres have identical medians because they contain exactly the same films in this candidate list. Kwame can treat them as one group of candidates when exploring the shortlist.

## Sensitivity analysis: missing records and repeated input rows

These local-cache scenarios use the same validated IDs, cached movie details,
`flatten_movie_details()` transform, and release windows as `run_analysis.py`.
The franchise statistic is the median **across films**, not the equally weighted
median across collections. The gap is franchise minus standalone median.

The missing-record scenarios assume all five unavailable IDs are standalone
films with usable positive financial values. Each receives either the lowest
or the highest observed usable standalone ratio **within that release window**.
For the 2000-onward scenarios, all five hypothetical films are explicitly assumed
to meet the release cutoff. These are separate scenarios under stated assumptions,
not recovered observations, predictions, or bounds for every possible missing film.

The repeated-row scenario adds one copy for each numeric row marked `duplicate`
in `id_audit.csv`, after applying the validator's trimming, positive-integer check,
and leading-zero normalization. It excludes rejected IMDb-format IDs and keeps
each actual film's release date and franchise membership. Scenarios are applied
separately; missing records and duplicate rows are not combined.


In [122]:
# Self-contained, read-only setup: no acquisition or output CSV writes.
import json
import sqlite3
import sys
from contextlib import closing
from pathlib import Path

import pandas as pd

sensitivity_root = Path.cwd()
if sensitivity_root.name.lower() == "data profile":
    sensitivity_root = sensitivity_root.parent
if str(sensitivity_root) not in sys.path:
    sys.path.insert(0, str(sensitivity_root))

from pipeline.analysis import analyze_licensing
from pipeline.checks import check_acquisition, check_movies
from pipeline.transform import flatten_movie_details

sensitivity_paths = {
    "validated": sensitivity_root / "data/validated/validated_ids.csv",
    "id_audit": sensitivity_root / "data/validated/id_audit.csv",
    "fetch_audit": sensitivity_root / "data/cache/fetch_audit.csv",
    "cache": sensitivity_root / "data/cache/tmdb.sqlite",
}
missing_files = [str(path) for path in sensitivity_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Sensitivity results unavailable; missing: " + ", ".join(missing_files))

sensitivity_validated = pd.read_csv(sensitivity_paths["validated"], dtype={"tmdb_id": str})
sensitivity_id_audit = pd.read_csv(
    sensitivity_paths["id_audit"], dtype={"tmdb_id": str}, keep_default_na=False,
)
sensitivity_fetch_audit = pd.read_csv(sensitivity_paths["fetch_audit"], dtype={"tmdb_id": str})
sensitivity_valid_ids = set(sensitivity_validated["tmdb_id"])
with closing(sqlite3.connect(sensitivity_paths["cache"].resolve().as_uri() + "?mode=ro", uri=True)) as db:
    sensitivity_rows = db.execute(
        "SELECT movie_id, endpoint, payload FROM responses ORDER BY movie_id"
    ).fetchall()

sensitivity_records = [
    {**json.loads(payload), "tmdb_id": movie_id}
    for movie_id, endpoint, payload in sensitivity_rows
    if endpoint == "details" and movie_id in sensitivity_valid_ids
]
sensitivity_movies = flatten_movie_details(pd.DataFrame(sensitivity_records))
check_movies(sensitivity_validated, sensitivity_records, sensitivity_movies)
check_acquisition(sensitivity_validated, sensitivity_rows, sensitivity_fetch_audit)
missing_record_ids = sensitivity_validated.loc[
    ~sensitivity_validated["tmdb_id"].isin(sensitivity_movies["tmdb_id"]), "tmdb_id"
].tolist()
assert len(missing_record_ids) == 5, "Revisit the five-missing-film assumption: cache coverage changed."

sensitivity_windows = {
    "All years": sensitivity_movies,
    "2000 onward": sensitivity_movies.loc[sensitivity_movies["release_date"].dt.year.ge(2000)],
}
pipeline_comparison = analyze_licensing(sensitivity_movies)["comparison"].set_index("time_window")
print(f"Cached films used: {len(sensitivity_movies)} of {len(sensitivity_valid_ids)} validated IDs")
print("Missing detail IDs:", ", ".join(missing_record_ids))
print("Details and credits audit failures:")
display(sensitivity_fetch_audit.loc[
    sensitivity_fetch_audit["tmdb_id"].isin(missing_record_ids), ["tmdb_id", "details", "credits"]
])


Cached films used: 112 of 117 validated IDs
Missing detail IDs: 8471293, 9412887, 8471294, 9412888, 7733901
Details and credits audit failures:


,tmdb_id,details,credits
16,8471293,8471293 details: HTTP 404,8471293 credits: HTTP 404
32,9412887,9412887 details: HTTP 404,9412887 credits: HTTP 404
67,8471294,8471294 details: HTTP 404,8471294 credits: HTTP 404
85,9412888,9412888 details: HTTP 404,9412888 credits: HTTP 404
89,7733901,7733901 details: HTTP 404,7733901 credits: HTTP 404


In [123]:
# Normalize duplicate IDs exactly as pipeline/validate.py does.
clean_audit_ids = sensitivity_id_audit["tmdb_id"].str.strip()
numeric_audit_ids = clean_audit_ids.str.fullmatch(r"[0-9]+") & ~clean_audit_ids.str.fullmatch(r"0+")
normalized_numeric_ids = clean_audit_ids.loc[numeric_audit_ids].str.lstrip("0")
expected_duplicate_rows = normalized_numeric_ids[normalized_numeric_ids.duplicated(keep="first")]
actual_duplicate_rows = sensitivity_id_audit["status"].eq("duplicate")
assert set(expected_duplicate_rows.index) == set(sensitivity_id_audit.index[actual_duplicate_rows])
repeat_ids = normalized_numeric_ids.loc[actual_duplicate_rows[actual_duplicate_rows].index]
assert len(repeat_ids) == 6, "Revisit the six-repeat assumption: ID audit changed."
assert repeat_ids.isin(sensitivity_movies["tmdb_id"]).all(), "A repeated ID lacks movie details."

repeat_films = sensitivity_movies.set_index("tmdb_id").loc[repeat_ids.tolist()].reset_index()
repeat_evidence = repeat_films[["tmdb_id", "title", "release_date", "collection_id", "revenue_to_budget"]].copy()
repeat_evidence.insert(0, "source_line", sensitivity_id_audit.loc[repeat_ids.index, "source_line"].to_numpy())
repeat_evidence["group"] = repeat_evidence["collection_id"].notna().map({True: "Franchise", False: "Standalone"})
repeat_evidence["meets_2000_cutoff"] = repeat_evidence["release_date"].dt.year.ge(2000)
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(repeat_evidence.round({"revenue_to_budget": 6}))
print("Rejected IMDb-format rows excluded:", sensitivity_id_audit.loc[
    sensitivity_id_audit["status"].eq("invalid_format") & clean_audit_ids.str.startswith("tt"), "tmdb_id"
].tolist())


,source_line,tmdb_id,title,release_date,collection_id,revenue_to_budget,group,meets_2000_cutoff
0,43,603,The Matrix,1999-03-31,2344,7.357419,Franchise,False
1,60,278,The Shawshank Redemption,1994-09-23,<NA>,1.133659,Standalone,False
2,90,1891,The Empire Strikes Back,1980-05-20,10,29.911111,Franchise,False
3,91,155,The Dark Knight,2008-07-16,263,5.451246,Franchise,True
4,94,550,Fight Club,1999-10-15,<NA>,1.600853,Standalone,False
5,102,680,Pulp Fiction,1994-09-10,<NA>,26.741095,Standalone,False


Rejected IMDb-format rows excluded: ['tt0468569']


In [124]:
sensitivity_results = []
scenario_inputs = []

for window, films in sensitivity_windows.items():
    franchise_ratios = films.loc[films["collection_id"].notna(), "revenue_to_budget"].dropna()
    standalone_ratios = films.loc[films["collection_id"].isna(), "revenue_to_budget"].dropna()
    if franchise_ratios.empty or standalone_ratios.empty:
        raise ValueError(f"Cannot calculate both medians for {window}")
    base_franchise = franchise_ratios.median()
    base_standalone = standalone_ratios.median()
    pipeline_window = "all_years" if window == "All years" else "2000_onward"
    assert base_franchise == pipeline_comparison.loc[pipeline_window, "franchise_film_median"]
    assert base_standalone == pipeline_comparison.loc[pipeline_window, "standalone_median"]

    window_repeats = repeat_films if window == "All years" else repeat_films.loc[
        repeat_films["release_date"].dt.year.ge(2000)
    ]
    repeated = pd.concat([films, window_repeats], ignore_index=True)
    repeated_franchise = repeated.loc[repeated["collection_id"].notna(), "revenue_to_budget"].median()
    repeated_standalone = repeated.loc[repeated["collection_id"].isna(), "revenue_to_budget"].median()
    low, high = standalone_ratios.min(), standalone_ratios.max()
    scenario_inputs.append({
        "window": window, "baseline_franchise_films": len(franchise_ratios),
        "baseline_standalone_films": len(standalone_ratios),
        "hypothetical_ratio_low": low, "hypothetical_ratio_high": high,
        "repeat_rows_in_window": len(window_repeats),
    })
    scenarios = [
        ("Baseline", base_franchise, base_standalone),
        ("Five missing: low standalone ratio", base_franchise,
         pd.concat([standalone_ratios, pd.Series([low] * len(missing_record_ids))], ignore_index=True).median()),
        ("Five missing: high standalone ratio", base_franchise,
         pd.concat([standalone_ratios, pd.Series([high] * len(missing_record_ids))], ignore_index=True).median()),
        ("Count repeated numeric rows again", repeated_franchise, repeated_standalone),
    ]
    for scenario, franchise, standalone in scenarios:
        gap = franchise - standalone
        sensitivity_results.append({
            "window": window, "scenario": scenario,
            "franchise_median": franchise, "standalone_median": standalone, "gap": gap,
            "franchise_change": franchise - base_franchise,
            "standalone_change": standalone - base_standalone,
            "leader": "Franchise" if gap > 0 else "Standalone" if gap < 0 else "Tie",
        })

sensitivity_inputs = pd.DataFrame(scenario_inputs)
sensitivity_table = pd.DataFrame(sensitivity_results)
with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_colwidth", None):
    print("Scenario inputs (unrounded values used in calculations):")
    display(sensitivity_inputs.round(6))
    print("Scenario medians, gap (franchise minus standalone), and changes from baseline:")
    display(sensitivity_table.set_index(["window", "scenario"]).round(6))


Scenario inputs (unrounded values used in calculations):


,window,baseline_franchise_films,baseline_standalone_films,hypothetical_ratio_low,hypothetical_ratio_high,repeat_rows_in_window
0,All years,96,16,1.133659,32.416667,6
1,2000 onward,63,8,1.642246,14.469742,1


Scenario medians, gap (franchise minus standalone), and changes from baseline:


franchise_median  standalone_median       gap  franchise_change  standalone_change      leader
window      scenario                                                                                                                           
All years   Baseline                                     5.623799           3.844812  1.778988          0.000000           0.000000   Franchise
            Five missing: low standalone ratio           5.623799           2.741908  2.881891          0.000000          -1.102904   Franchise
            Five missing: high standalone ratio          5.623799           5.243941  0.379858          0.000000           1.399130   Franchise
            Count repeated numeric rows again            5.721205           3.238500  2.482705          0.097406          -0.606312   Franchise
2000 onward Baseline                                     5.051939           3.844812  1.207127          0.000000           0.000000   Franchise
            Five missing: low standalone ratio           5.051939           2.741908  2.310031          0.000000          -1.102904   Franchise
            Five missing: high standalone ratio          5.051939           5.243941 -0.192003          0.000000           1.399130  Standalone
            Count repeated numeric rows again            5.061960           3.844812  1.217149          0.010022           0.000000   Franchise

In [125]:
# Recheck the notebook's financial-quality observations on pipeline movie data.
financial_checks = pd.Series({
    "Films checked": len(sensitivity_movies),
    "Budget missing or non-numeric": sensitivity_movies["budget"].isna().sum(),
    "Budget is zero": sensitivity_movies["budget"].eq(0).sum(),
    "Revenue missing or non-numeric": sensitivity_movies["revenue"].isna().sum(),
    "Revenue is zero": sensitivity_movies["revenue"].eq(0).sum(),
    "Negative budgets": sensitivity_movies["budget"].lt(0).sum(),
    "Negative revenues": sensitivity_movies["revenue"].lt(0).sum(),
    "Release date missing or invalid": sensitivity_movies["release_date"].isna().sum(),
    "Usable revenue-to-budget ratios": sensitivity_movies["revenue_to_budget"].notna().sum(),
    "Ratio above 20 (review threshold, not an exclusion)": sensitivity_movies["revenue_to_budget"].gt(20).sum(),
}, name="films")
display(financial_checks.to_frame())
print("Five highest cached ratios (retained in analysis):")
display(sensitivity_movies.nlargest(5, "revenue_to_budget")[[
    "tmdb_id", "title", "budget", "revenue", "revenue_to_budget"
]].round({"revenue_to_budget": 6}))
print("Figures have no inflation adjustment. Grosses can include re-releases; the cache does not establish a film-specific re-release contribution.")


,films
Films checked,112
Budget missing or non-numeric,0
Budget is zero,0
Revenue missing or non-numeric,0
Revenue is zero,0
Negative budgets,0
Negative revenues,0
Release date missing or invalid,0
Usable revenue-to-budget ratios,112
"Ratio above 20 (review threshold, not an exclusion)",8


Five highest cached ratios (retained in analysis):


,tmdb_id,title,budget,revenue,revenue_to_budget
5,11,Star Wars,11000000,775398007,70.490728
73,578,Jaws,7000000,470653000,67.236143
38,238,The Godfather,6000000,245066411,40.844402
65,429,"The Good, the Bad and the Ugly",1200000,38900000,32.416667
29,1891,The Empire Strikes Back,18000000,538400000,29.911111


Figures have no inflation adjustment. Grosses can include re-releases; the cache does not establish a film-specific re-release contribution.


### Measured result and decision implications

The 112 cached films reproduce the pipeline baseline: franchise versus standalone
medians are **5.624 vs 3.845** across all years (96 vs 16 films) and **5.052 vs 3.845**
from 2000 onward (63 vs 8). All medians use unrounded ratios.

For the five missing-film scenarios, all-years standalone ratios are set to
1.133659 (low) or 32.416667 (high); the 2000-onward inputs are 1.642246 or 14.469742.
The low scenarios reduce the standalone median to **2.742** in both windows;
the high scenarios raise it to **5.244**. Franchises still lead in both all-years
scenarios. The high 2000-onward scenario reverses the lead: standalones lead
by **0.192**, assuming all five missing films are standalone and meet the cutoff.
The low 2000-onward scenario preserves the franchise lead. This is conditional
sensitivity evidence, not a claim about what those missing films actually are.

Counting the six repeated numeric rows again gives all-years medians of
**5.721 vs 3.239**, changes of **+0.097** and **-0.606** from baseline. Only the repeated
The Dark Knight record meets the 2000 cutoff; that window becomes **5.062 vs 3.845**,
changes of **+0.010** and **0.000**. Neither duplicate scenario changes the leader.
The rejected IMDb-format row is excluded. Repeats overweight existing films
rather than add film coverage, which supports keeping the pipeline deduplication.

The financial checks find zero missing/non-numeric, zero, or negative budgets
and revenues and zero invalid release dates among 112 films; all 112 ratios are
usable. Eight ratios exceed the explicit 20-times review threshold. The five
highest are Star Wars (70.491), Jaws (67.236), The Godfather (40.844),
The Good, the Bad and the Ugly (32.417), and The Empire Strikes Back (29.911).
They remain in the analysis: a high ratio alone does not establish an error.

Kwame has evidence to prioritize franchise discussions, but the small recent
standalone sample and missing-record scenarios mean the recent advantage is
conditional. Use title-level licensing terms and expected streaming demand
before deciding. Figures have no inflation adjustment, and grosses can include
re-releases. No film-specific re-release contribution is inferred from the cache;
revenue relative to production budget is not net profit or streaming return.
